# Low-Pass Butterworth Filter for EEG

Applies a 40 Hz low-pass filter to the output of the high-pass filter.

**Dataset**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 200 Hz

## 1. Install dependencies

In [ ]:
!pip install scipy numpy pandas matplotlib wfdb

## 2. Clone the resources repo

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')

## 3. Download the local EEG dataset

In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.csv')):
    !python data/download_local.py --output data/local

## 4. Apply high-pass then low-pass filter

In [ ]:
from scipy import signal
import numpy as np
import matplotlib.pyplot as plt
from utils.eeg_loader import load_local_eeg

# Load local EEG data
timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
channel_data = eeg_data[:, 0]  # P4 channel
fs = 200  # Sampling rate (Hz)

# High-pass filter at 1 Hz
def butter_highpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = signal.butter(order, normal_cutoff, btype='high', analog=False)
    return signal.filtfilt(b, a, data)

# Low-pass filter at 40 Hz
def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = signal.butter(order, normal_cutoff, btype='low', analog=False)
    return signal.filtfilt(b, a, data)

filtered_hp = butter_highpass_filter(channel_data, cutoff=1.0, fs=fs)
filtered_lp = butter_lowpass_filter(filtered_hp, cutoff=40.0, fs=fs)

# Plot before and after low-pass
n_plot = 5000
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(timestamps[:n_plot], filtered_hp[:n_plot], label='After HP only')
axes[0].set_ylabel('EEG (uV)')
axes[0].set_title('Before low-pass filter')
axes[1].plot(timestamps[:n_plot], filtered_lp[:n_plot], label='After LP', color='red')
axes[1].set_ylabel('EEG (uV)')
axes[1].set_xlabel('Time (ms)')
axes[1].set_title('After low-pass filter (40 Hz)')
plt.tight_layout()
plt.savefig('lowpass_result.png', dpi=150)
plt.show()